# 🌱 Stretch · Agent memory & context engineering

**Self-paced**

The `run_agent` loop forgets everything between calls. Real assistants need **memory**.
Here we add a simple running memory and think about *context engineering* — deciding what
to put in the model's limited context window.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from shared.llm import chat

class Memory:
    """A tiny rolling memory of facts ARIA has learned this session."""
    def __init__(self, max_items=8):
        self.facts = []
        self.max_items = max_items
    def remember(self, fact):
        self.facts.append(fact)
        self.facts = self.facts[-self.max_items:]  # keep context small
    def as_context(self):
        return "\n".join(f"- {f}" for f in self.facts) or "(nothing yet)"

In [ ]:
mem = Memory()
mem.remember("The Module B scrubber has tripped 3 times this month.")
mem.remember("Crew prefers relocating to Module A during O2 events.")

def ask_with_memory(question):
    system = (
        "You are ARIA. Use the following remembered facts if relevant.\n"
        f"MEMORY:\n{mem.as_context()}"
    )
    return chat(question, system=system)

print(ask_with_memory("If the scrubber trips again, what do you recommend and why?"))

### Context engineering questions

- Context windows are finite. What do you keep, summarize, or drop?
- Where should long-term memory live? (A vector store — like Module 4 — is a common answer:
  embed past events and retrieve only the relevant ones.)
- Too much irrelevant context *hurts* answers. Less, but relevant, is usually better.